# Sistema de Gestión - Estructura de Desarrollo (Arquitectura CRUD)

## Objetivo

Construir el sistema siguiendo una arquitectura por capas:

```text
VIEW (NiceGUI)
      │
      ▼
CONTROLLER
      │
      ▼
SERVICE
      │
      ▼
REPOSITORY
      │
      ▼
SQLAlchemy Session
      │
      ▼
Base de Datos
```

Cada módulo del sistema seguirá esta misma estructura.

---

# FASE 1 - Modelado de Base de Datos

## Crear BaseModel

```
models/
    base.py
```

Contiene:

- id
- Base Declarative

---

## Crear modelos

```
Usuario
Recipiente
Transaccion
DetalleTransaccion
```

Agregar:

- __tablename__
- Columnas
- ForeignKey
- relationship()
- back_populates

Registrar todos los modelos en:

```
models/__init__.py
```

---

# FASE 2 - Repository

Cada entidad tendrá su Repository.

```
repositories/

    base_repository.py

    usuario_repository.py

    recipiente_repository.py

    transaccion_repository.py

    detalle_transaccion_repository.py
```

El Repository únicamente interactúa con la base de datos.

Funciones típicas:

```
add()

update()

delete()

get_by_id()

get_all()

commit()

rollback()

flush()

refresh()
```

No contiene reglas de negocio.

---

# FASE 3 - Service

Cada módulo tiene su Service.

```
services/

    usuario_service.py

    recipiente_service.py

    transaccion_service.py
```

Aquí vive toda la lógica.

Ejemplos:

Usuario

- validar correo
- validar nombre

Recipiente

- validar precio
- validar inventario

Transacción

- crear cabecera
- crear detalles
- descontar inventario
- validar existencias

El Service nunca conoce NiceGUI.

---

# FASE 4 - Controller

```
controllers/

    usuario_controller.py

    recipiente_controller.py

    transaccion_controller.py
```

Su trabajo es conectar:

```
VIEW
   │
   ▼
Controller
   │
   ▼
Service
```

Generalmente:

```
with SessionLocal() as session:

    service = ...

    return service....
```

---

# FASE 5 - Views NiceGUI

```
views/

    usuarios.py

    recipientes.py

    transacciones.py
```

La View únicamente:

- muestra información
- captura datos
- llama al Controller
- muestra mensajes

Nunca consulta la base directamente.

---

# CRUD Usuario

## CREATE

```
Formulario

↓

UsuarioController

↓

UsuarioService

↓

UsuarioRepository

↓

Base de datos
```

Guardar:

```
Nombre

Correo

Teléfono

Activo
```

---

## READ

Mostrar listado

```
Repository.get_all()
```

Buscar por ID

```
Repository.get_by_id()
```

Buscar por nombre

```
Repository.get_by_nombre()
```

---

## UPDATE

```
Buscar Usuario

↓

Modificar campos

↓

Repository.update()

↓

commit()
```

El ID nunca cambia.

Las transacciones continúan relacionadas.

---

## DELETE

No eliminar físicamente.

Mejor:

```
activo=False
```

---

# CRUD Recipiente

## CREATE

Guardar

```
Nombre

Precio

Cantidad disponible

Activo
```

---

## READ

Listado

Buscar

Filtros

---

## UPDATE

Modificar

```
Nombre

Precio

Cantidad

Estado
```

Las transacciones siguen funcionando porque usan:

```
recipiente_id
```

---

## DELETE

Preferiblemente

```
activo=False
```

---

# CRUD Transaccion

No es un CRUD tradicional.

Representa un movimiento.

---

## CREATE

```
Usuario

+

Uno o varios recipientes

↓

TransaccionService

↓

Crear Transaccion

↓

Crear Detalles

↓

Actualizar Inventario

↓

Commit
```

---

## READ

Historial

```
Fecha

Usuario

Total

Estado
```

Detalle

```
Recipiente

Cantidad

Precio
```

---

## UPDATE

No modificar directamente.

Crear un proceso:

```
Editar Transacción

↓

Revertir Inventario

↓

Modificar Detalles

↓

Actualizar Inventario
```

---

## DELETE

No eliminar.

Usar:

```
Estado

CREADA

ANULADA
```

Al anular:

```
Devolver inventario
```

---

# CRUD DetalleTransaccion

Normalmente no existe un formulario independiente.

Pertenece a una Transacción.

---

## CREATE

Se crea desde:

```
TransaccionService
```

---

## READ

Mostrar detalles de una transacción.

---

## UPDATE

Permitir corregir:

```
Cantidad

Precio
```

Ajustando inventario.

---

## DELETE

Eliminar detalle

↓

Devolver inventario

↓

Commit

---

# Flujo General

```text
NiceGUI

↓

Controller

↓

Service

↓

Repository

↓

SQLAlchemy

↓

Base de Datos
```

---

# Orden recomendado de desarrollo

## Paso 1

Modelo Base

```
BaseModel
```

---

## Paso 2

Modelos

```
Usuario

Recipiente

Transaccion

DetalleTransaccion
```

---

## Paso 3

Repositories

```
BaseRepository

UsuarioRepository

RecipienteRepository

TransaccionRepository

DetalleRepository
```

---

## Paso 4

CRUD Usuario

- Crear
- Buscar
- Listar
- Editar
- Desactivar

---

## Paso 5

CRUD Recipiente

- Crear
- Buscar
- Listar
- Editar
- Inventario

---

## Paso 6

CRUD Transacción

- Crear
- Historial
- Consultar
- Anular

---

## Paso 7

DetalleTransacción

- Consultar
- Corregir
- Eliminar detalle

---

## Paso 8

Mejoras

- Login
- Roles
- Permisos
- Reportes
- Dashboard
- Auditoría
- Logs
- Alembic (Migraciones)
- Validaciones avanzadas